In [ ]:
import torch
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
# from utils import (
#   load_checkpoint,
#   save_checkpoint,
#   get_loaders,
#   check_accuracy,
#   save_predictions_as_imgs
# )

In [ ]:
@dataclass
class TrainerConfig:
  learning_rate: float = 1e-4
  device: str = "cuda" if torch.cuda.is_available() else "cpu"
  batch_size: int = 32
  num_epochs: int = 3
  num_workers: int = 2
  image_height: int = 160
  image_width: int = 240
  pin_memory: bool = True
  load_model: bool = False

In [ ]:
class Trainer():
  def __init__(self, loader, model, optimizer, loss_fn, scaler, config=TrainerConfig()):
    self.loader = loader
    self.model = model
    self.optimizer = optimizer
    self.loss_fn = loss_fn
    self.scaler = scaler
    self.config = config
    
  def __call__(self):
    self.step()
  
  def step(self):
    loop = tqdm(self.loader)
    
    for batch_idx, (data, targets) in enumerate(self.loop):
      data = data.to(device=self.config.device)
      targets = targets.float().unsqueeze(1).to(device=self.config.device)
      
      # forward - Flow-16
      with torch.cuda.amp.autocast():
        predictions = self.model(data)
        loss = self.loss_fn(predictions, targets)
        
      # backward
      self.optimizer.zero_grad()
      self.scaler.scale(loss).backward()
      self.scaler.step(self.optimizer)
      self.scaler.update()
      
      # update tqdm loop
      self.loop.set_postfix(loss=self.loss.item())
      

2
